# RX Strategist Streamlit (Colab)

Launch the dashboard from Colab. Add `GEMINI_API_KEY` to **Colab Secrets** first.

This notebook clones the repo, installs dependencies, starts Streamlit on port 8501, and opens it through Colab's port preview.

The Streamlit code must be on GitHub (`app.py` at the repo root). Push your latest dashboard commit before running this notebook.

In [ ]:
import os
import sys
from pathlib import Path

REPO_URL = "https://github.com/Anagha-Krish-P/rx-strategist-mvp.git"

def find_src():
    for path in (Path("src"), Path("rx-strategist-mvp/src"), Path("../src")):
        if (path / "rx_strategist").is_dir():
            return path.resolve()
    return None

src = find_src()
if src is None:
    get_ipython().system(f"git clone {REPO_URL}")
    src = find_src()

if src is None:
    raise FileNotFoundError("Could not find src/rx_strategist. Clone failed or the repo is missing.")

REPO_ROOT = src.parent
os.chdir(REPO_ROOT)
sys.path.insert(0, str(src))
get_ipython().run_line_magic("pip", f"install -q -r {REPO_ROOT / 'requirements.txt'}")
print("Repo root:", REPO_ROOT)
print("app.py exists:", (REPO_ROOT / "app.py").is_file())

In [ ]:
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
if not os.environ["GEMINI_API_KEY"]:
    raise ValueError("Add GEMINI_API_KEY to Colab Secrets, then re-run this cell.")
print("GEMINI_API_KEY loaded.")

In [ ]:
import subprocess
import time
import urllib.request

from google.colab import output

PORT = 8501
get_ipython().system(f"fuser -k {PORT}/tcp >/dev/null 2>&1 || true")

proc = subprocess.Popen(
    [
        "streamlit",
        "run",
        "app.py",
        "--server.port",
        str(PORT),
        "--server.address",
        "0.0.0.0",
        "--server.headless",
        "true",
        "--server.enableCORS",
        "false",
        "--server.enableXsrfProtection",
        "false",
    ],
    cwd=str(REPO_ROOT),
)

for _ in range(40):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/_stcore/health")
        break
    except Exception:
        time.sleep(0.5)
else:
    raise RuntimeError("Streamlit did not start. Re-run this cell.")

print("Streamlit is running. Opening the Colab port preview...")
output.serve_kernel_port_as_window(PORT)